# Module 1: EIA Petroleum Report Extraction

## Learning Objectives

By completing this notebook, you will:
1. Download and parse EIA Weekly Petroleum Status Reports (PDFs)
2. Extract structured data from both tables and narrative text
3. Use LLMs to identify and extract key market-moving information
4. Build a time series database of inventory changes
5. Implement quality control checks for production systems

## Prerequisites

- Completed Module 0, Notebook 2 (LLM Basics)
- OpenAI or Anthropic API key configured
- Understanding of petroleum markets (crude, gasoline, distillate)
- Familiarity with pandas DataFrame operations

## Estimated Time: 75-90 minutes

---

## 1. Setup and Configuration

The EIA releases the Weekly Petroleum Status Report every Wednesday at 10:30 AM ET. This report moves oil markets immediately upon release. We'll build a system to automatically extract and structure the data.

**Why This Matters:**
- Reports are PDFs, not machine-readable
- Critical data is in both tables and text
- Speed matters - markets react in seconds
- Historical backfill enables strategy testing

In [ ]:
# Standard library imports
import os
import json
import re
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import requests
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns

# PDF processing
import PyPDF2
import pdfplumber

# LLM imports
import openai
from anthropic import Anthropic

# Configuration
load_dotenv()
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# API credentials
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

# Initialize LLM clients
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

# Create directories for data storage
DATA_DIR = Path('data/eia_reports')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete")
print(f"   Data directory: {DATA_DIR.absolute()}")
print(f"   LLM clients: {'OpenAI' if openai_client else ''} {'Claude' if anthropic_client else ''}")

## 2. Downloading EIA Reports

EIA reports are available at predictable URLs. Each Wednesday's report follows the pattern:
`https://www.eia.gov/petroleum/supply/weekly/archive/YYYY/YYYY_MM_DD.pdf`

**Key Considerations:**
- Reports are published at 10:30 AM ET
- Archives go back to 1990s
- File sizes are 200-500 KB
- Need to handle 404s for holidays/missing weeks

In [ ]:
def generate_report_url(report_date: datetime) -> str:
    """
    Generate EIA report URL for a given date.
    
    Parameters:
    -----------
    report_date : datetime
        Date of the report (should be a Wednesday)
        
    Returns:
    --------
    str
        URL to the PDF report
    """
    year = report_date.year
    date_str = report_date.strftime('%Y_%m_%d')
    
    url = f"https://www.eia.gov/petroleum/supply/weekly/archive/{year}/{date_str}.pdf"
    return url

def download_report(report_date: datetime, save_dir: Path = DATA_DIR) -> Optional[Path]:
    """
    Download EIA petroleum report for a specific date.
    
    Parameters:
    -----------
    report_date : datetime
        Date of the report to download
    save_dir : Path
        Directory to save the PDF
        
    Returns:
    --------
    Optional[Path]
        Path to downloaded file, or None if failed
    """
    url = generate_report_url(report_date)
    filename = f"eia_petroleum_{report_date.strftime('%Y_%m_%d')}.pdf"
    filepath = save_dir / filename
    
    # Check if already downloaded
    if filepath.exists():
        print(f"✅ Report already exists: {filename}")
        return filepath
    
    # Download report
    try:
        print(f"Downloading {filename}...")
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        # Save to file
        filepath.write_bytes(response.content)
        print(f"✅ Downloaded: {filename} ({len(response.content) / 1024:.1f} KB)")
        return filepath
        
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            print(f"⚠️  Report not found (404): {report_date.strftime('%Y-%m-%d')}")
        else:
            print(f"❌ HTTP error {e.response.status_code}: {url}")
        return None
    except Exception as e:
        print(f"❌ Download failed: {str(e)}")
        return None

# Download the most recent report (from last Wednesday)
# Find last Wednesday
today = datetime.now()
days_since_wednesday = (today.weekday() - 2) % 7  # Wednesday is 2
if days_since_wednesday == 0 and today.hour < 11:  # Before 11 AM on Wednesday
    days_since_wednesday = 7
last_wednesday = today - timedelta(days=days_since_wednesday)

print(f"\nAttempting to download report from {last_wednesday.strftime('%Y-%m-%d')}...\n")
report_path = download_report(last_wednesday)

if report_path:
    print(f"\n✅ Report ready for processing: {report_path.name}")
else:
    print("\n⚠️  Could not download latest report. Using sample data for demonstration.")

### 💡 Exercise 2.1: Batch Download Historical Reports

**Task:** Download the last 4 weeks of EIA reports.

**Requirements:**
1. Generate dates for the last 4 Wednesdays
2. Download each report using the `download_report()` function
3. Track success/failure for each download
4. Print summary statistics

**Hints:**
- Start from `last_wednesday` and go back in 7-day increments
- Some weeks may be missing (holidays)
- Store results in a list

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------
print("Downloading last 4 weeks of reports:\n")

download_results = []

for i in range(4):
    report_date = last_wednesday - timedelta(days=7*i)
    filepath = download_report(report_date)
    
    download_results.append({
        'date': report_date,
        'success': filepath is not None,
        'path': filepath
    })
    print()  # Blank line between downloads

# Summary
success_count = sum(1 for r in download_results if r['success'])
print("="*60)
print(f"Download Summary: {success_count}/{len(download_results)} successful")
print("="*60)

for result in download_results:
    status = "✅" if result['success'] else "❌"
    print(f"{status} {result['date'].strftime('%Y-%m-%d')}")

## 3. Extracting Text from PDFs

PDFs are not structured data formats. We need to extract text while preserving structure. Two main approaches:
1. **PyPDF2**: Simple text extraction
2. **pdfplumber**: Better table extraction, preserves layout

**EIA Report Structure:**
- Page 1: Summary statistics table
- Page 2-3: Narrative highlights
- Remaining pages: Detailed tables

In [ ]:
def extract_text_from_pdf(pdf_path: Path) -> Dict[str, Any]:
    """
    Extract text and tables from EIA PDF report.
    
    Parameters:
    -----------
    pdf_path : Path
        Path to PDF file
        
    Returns:
    --------
    dict
        Dictionary with 'pages', 'full_text', 'tables', 'summary_table'
    """
    result = {
        'pages': [],
        'full_text': '',
        'tables': [],
        'summary_table': None
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            print(f"Processing PDF: {pdf_path.name}")
            print(f"  Pages: {len(pdf.pages)}")
            
            for i, page in enumerate(pdf.pages):
                # Extract text
                text = page.extract_text()
                result['pages'].append({
                    'page_num': i + 1,
                    'text': text
                })
                result['full_text'] += f"\n--- Page {i+1} ---\n{text}"
                
                # Extract tables
                tables = page.extract_tables()
                for j, table in enumerate(tables):
                    if table:  # Not empty
                        result['tables'].append({
                            'page': i + 1,
                            'table_num': j + 1,
                            'data': table
                        })
                        
                        # First page, first table is usually the summary
                        if i == 0 and j == 0:
                            result['summary_table'] = table
            
            print(f"  ✅ Extracted {len(result['tables'])} tables")
            print(f"  ✅ Total text length: {len(result['full_text'])} characters")
            
    except Exception as e:
        print(f"❌ Error extracting PDF: {str(e)}")
    
    return result

# Extract text from our downloaded report
if report_path and report_path.exists():
    extracted = extract_text_from_pdf(report_path)
    
    # Display first 1000 characters
    print("\nFirst 1000 characters of extracted text:")
    print("="*60)
    print(extracted['full_text'][:1000])
    print("="*60)
else:
    print("⚠️  No report available to extract")
    extracted = None

## 4. Parsing the Summary Table

The first table in the EIA report contains the critical numbers:
- Current week values
- Week-over-week changes
- Year-ago comparisons

**Challenge:** Table structure varies slightly across report versions. We need robust parsing.

In [ ]:
def parse_summary_table(table_data: List[List[str]]) -> pd.DataFrame:
    """
    Parse the EIA summary table into structured DataFrame.
    
    Parameters:
    -----------
    table_data : List[List[str]]
        Raw table data from pdfplumber
        
    Returns:
    --------
    pd.DataFrame
        Cleaned and structured data
    """
    if not table_data:
        return pd.DataFrame()
    
    # Convert to DataFrame
    df = pd.DataFrame(table_data)
    
    # First row is usually headers
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    
    # Clean numeric columns
    for col in df.columns:
        if col and col not in ['Product', 'Item', 'Category']:
            # Remove commas and convert to numeric
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(',', '').str.replace('--', '').str.strip(),
                errors='coerce'
            )
    
    return df

# Parse the summary table
if extracted and extracted['summary_table']:
    summary_df = parse_summary_table(extracted['summary_table'])
    
    print("Parsed Summary Table:")
    print(summary_df.head(10))
    print(f"\nShape: {summary_df.shape}")
else:
    print("⚠️  No summary table to parse")
    summary_df = None

### 💡 Exercise 4.1: Extract Key Metrics from Table

**Task:** From the summary table, extract crude oil, gasoline, and distillate inventory levels and changes.

**Requirements:**
1. Find rows containing "Commercial Crude Oil", "Motor Gasoline", "Distillate Fuel"
2. Extract current level and weekly change columns
3. Create a clean summary DataFrame with just these key metrics
4. Handle missing data appropriately

**Hints:**
- Use string matching on the first column
- Check for variations in product names
- Column names may vary by report version

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------
if summary_df is not None and not summary_df.empty:
    # Define products of interest
    products = {
        'crude_oil': ['Commercial Crude Oil', 'Crude Oil', 'Commercial Crude'],
        'gasoline': ['Motor Gasoline', 'Total Gasoline', 'Gasoline'],
        'distillate': ['Distillate Fuel', 'Distillate', 'Distillate Fuel Oil']
    }
    
    key_metrics = []
    
    # Identify the product name column (usually first column)
    product_col = summary_df.columns[0]
    
    for product_key, search_terms in products.items():
        # Find matching row
        for term in search_terms:
            matches = summary_df[summary_df[product_col].astype(str).str.contains(term, case=False, na=False)]
            if not matches.empty:
                row = matches.iloc[0]
                
                # Extract available numeric columns
                metric = {
                    'product': product_key,
                    'product_name': row[product_col]
                }
                
                # Add all numeric columns
                for col in summary_df.columns[1:]:
                    if pd.api.types.is_numeric_dtype(summary_df[col]):
                        metric[col] = row[col]
                
                key_metrics.append(metric)
                break
    
    if key_metrics:
        key_metrics_df = pd.DataFrame(key_metrics)
        print("Key Inventory Metrics:")
        print(key_metrics_df)
        print(f"\nExtracted {len(key_metrics_df)} products")
    else:
        print("⚠️  Could not extract key metrics from table")
        key_metrics_df = None
else:
    print("⚠️  No summary table available")
    key_metrics_df = None

## 5. Using LLM for Narrative Extraction

Tables don't tell the full story. The narrative text contains:
- Context and explanations
- Comparisons to forecasts
- Forward-looking indicators
- Special events (refinery outages, hurricanes)

**LLM Advantage:** Can extract semi-structured insights that humans focus on.

In [ ]:
def extract_narrative_insights(text: str) -> dict:
    """
    Use LLM to extract insights from narrative text.
    
    Parameters:
    -----------
    text : str
        Full text of the EIA report
        
    Returns:
    --------
    dict
        Structured insights from the narrative
    """
    prompt = f"""
Analyze this EIA Weekly Petroleum Status Report and extract key insights.

Return JSON with the following structure:
{{
  "headline": "Brief one-sentence summary of the most important finding",
  "inventory_drivers": ["List of factors mentioned as causing inventory changes"],
  "market_context": "Any mentions of market conditions, prices, or expectations",
  "production_changes": "Any notable production changes or trends",
  "demand_signals": "Any indicators of demand strength or weakness",
  "special_events": ["List any special events: refinery maintenance, weather, etc."],
  "forward_indicators": "Any forward-looking statements or indicators",
  "surprises": ["List any values that differed significantly from typical patterns"]
}}

Only include information explicitly stated in the report. Use null for missing information.

Report Text:
{text[:8000]}  # Limit to first 8000 chars to manage token count
"""
    
    if openai_client:
        response = openai_client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": "You are a commodity market analyst. Extract facts, not opinions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    
    elif anthropic_client:
        message = anthropic_client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=2000,
            temperature=0.0,
            messages=[{"role": "user", "content": prompt}]
        )
        response_text = message.content[0].text
        if "```json" in response_text:
            response_text = response_text.split("```json")[1].split("```")[0]
        return json.loads(response_text.strip())
    
    else:
        return {"error": "No LLM client configured"}

# Extract insights from narrative
if extracted and extracted['full_text']:
    print("Extracting narrative insights with LLM...\n")
    insights = extract_narrative_insights(extracted['full_text'])
    
    print("Extracted Insights:")
    print(json.dumps(insights, indent=2))
else:
    print("⚠️  No text available for narrative extraction")
    insights = None

### 💡 Exercise 5.1: Insight Quality Assessment

**Task:** Evaluate the quality and usefulness of the LLM-extracted insights.

**Requirements:**
1. Read the original report text (first 2 pages)
2. Compare LLM insights against what you read
3. Identify any hallucinations or missing key points
4. Rate each insight field as: accurate, partially accurate, or inaccurate
5. Suggest improvements to the prompt

**This is a manual exercise - write your assessment below.**

### Your Assessment:

**Headline Accuracy:**
- [Your assessment here]

**Inventory Drivers:**
- [Your assessment here]

**Missing Information:**
- [What did the LLM miss?]

**Hallucinations:**
- [Any invented facts?]

**Prompt Improvements:**
- [How would you improve the prompt?]

## 6. Building a Complete Extraction Pipeline

Let's combine everything into a production-ready pipeline that:
1. Downloads reports
2. Extracts text and tables
3. Parses structured data
4. Extracts LLM insights
5. Combines into unified output
6. Validates results

In [ ]:
def process_eia_report(report_date: datetime) -> dict:
    """
    Complete pipeline for processing EIA report.
    
    Parameters:
    -----------
    report_date : datetime
        Date of report to process
        
    Returns:
    --------
    dict
        Complete structured output
    """
    result = {
        'report_date': report_date.strftime('%Y-%m-%d'),
        'processed_at': datetime.now().isoformat(),
        'success': False,
        'data': None,
        'insights': None,
        'errors': []
    }
    
    try:
        # Step 1: Download
        print(f"[1/5] Downloading report for {report_date.strftime('%Y-%m-%d')}...")
        pdf_path = download_report(report_date)
        if not pdf_path:
            result['errors'].append("Download failed")
            return result
        
        # Step 2: Extract text/tables
        print("[2/5] Extracting text and tables...")
        extracted = extract_text_from_pdf(pdf_path)
        if not extracted['full_text']:
            result['errors'].append("PDF extraction failed")
            return result
        
        # Step 3: Parse summary table
        print("[3/5] Parsing summary table...")
        if extracted['summary_table']:
            summary_df = parse_summary_table(extracted['summary_table'])
            result['data'] = summary_df.to_dict('records')
        else:
            result['errors'].append("No summary table found")
        
        # Step 4: Extract LLM insights
        print("[4/5] Extracting narrative insights...")
        insights = extract_narrative_insights(extracted['full_text'])
        result['insights'] = insights
        
        # Step 5: Validate
        print("[5/5] Validating results...")
        if result['data'] and result['insights']:
            result['success'] = True
            print("✅ Pipeline complete!")
        else:
            result['errors'].append("Incomplete data extraction")
        
    except Exception as e:
        result['errors'].append(f"Pipeline error: {str(e)}")
        print(f"❌ Pipeline failed: {str(e)}")
    
    return result

# Process the latest report
print("Running complete extraction pipeline:\n")
print("="*60)
pipeline_result = process_eia_report(last_wednesday)
print("="*60)

# Display results summary
print(f"\nPipeline Result:")
print(f"  Success: {pipeline_result['success']}")
print(f"  Data records: {len(pipeline_result['data']) if pipeline_result['data'] else 0}")
print(f"  Insights extracted: {bool(pipeline_result['insights'])}")
if pipeline_result['errors']:
    print(f"  Errors: {pipeline_result['errors']}")

## 7. Storing Extracted Data

For production use, we need to store extracted data in a queryable format. Options:
- CSV files (simple, human-readable)
- SQLite database (queryable, relational)
- Time series database (InfluxDB, TimescaleDB)

We'll use CSV for simplicity, but structure for easy database migration.

In [ ]:
def save_extraction_results(result: dict, output_dir: Path = DATA_DIR):
    """
    Save extraction results to files.
    
    Parameters:
    -----------
    result : dict
        Output from process_eia_report()
    output_dir : Path
        Directory to save results
    """
    report_date = result['report_date']
    
    # Save structured data
    if result['data']:
        data_file = output_dir / f"eia_data_{report_date}.csv"
        df = pd.DataFrame(result['data'])
        df['report_date'] = report_date
        df.to_csv(data_file, index=False)
        print(f"✅ Saved data: {data_file.name}")
    
    # Save insights
    if result['insights']:
        insights_file = output_dir / f"eia_insights_{report_date}.json"
        with open(insights_file, 'w') as f:
            json.dump(result['insights'], f, indent=2)
        print(f"✅ Saved insights: {insights_file.name}")
    
    # Save metadata
    metadata_file = output_dir / f"eia_metadata_{report_date}.json"
    metadata = {
        'report_date': report_date,
        'processed_at': result['processed_at'],
        'success': result['success'],
        'errors': result['errors']
    }
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"✅ Saved metadata: {metadata_file.name}")

# Save our results
if pipeline_result['success']:
    save_extraction_results(pipeline_result)
else:
    print("⚠️  Skipping save - extraction was not successful")

### 💡 Exercise 7.1: Build Historical Database

**Task:** Process the 4 reports downloaded earlier and combine into a single time series database.

**Requirements:**
1. Process each of the 4 downloaded reports
2. Extract the structured data from each
3. Combine into a single DataFrame with date index
4. Create a visualization showing crude oil inventory trends
5. Save the combined database to CSV

**Expected Output:** Time series chart and combined CSV file

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------
print("Building historical database:\n")

all_results = []

for i in range(4):
    report_date = last_wednesday - timedelta(days=7*i)
    print(f"\nProcessing {report_date.strftime('%Y-%m-%d')}...")
    
    result = process_eia_report(report_date)
    if result['success']:
        all_results.append(result)
        save_extraction_results(result)

# Combine all data
if all_results:
    combined_data = []
    for result in all_results:
        if result['data']:
            df = pd.DataFrame(result['data'])
            df['report_date'] = result['report_date']
            combined_data.append(df)
    
    if combined_data:
        historical_df = pd.concat(combined_data, ignore_index=True)
        historical_df['report_date'] = pd.to_datetime(historical_df['report_date'])
        
        # Save combined database
        combined_file = DATA_DIR / 'eia_historical_combined.csv'
        historical_df.to_csv(combined_file, index=False)
        print(f"\n✅ Saved combined database: {combined_file}")
        print(f"   Total records: {len(historical_df)}")
        print(f"   Date range: {historical_df['report_date'].min()} to {historical_df['report_date'].max()}")
        
        # Visualize trends (if we have enough data)
        print("\nCreating trend visualization...")
        plt.figure(figsize=(14, 8))
        
        # Try to plot key inventory trends
        print("\nAvailable columns:", historical_df.columns.tolist())
        
        plt.title('EIA Petroleum Historical Data', fontsize=14, fontweight='bold')
        plt.xlabel('Report Date', fontsize=12)
        plt.ylabel('Value', fontsize=12)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
    else:
        print("\n⚠️  No data to combine")
else:
    print("\n⚠️  No successful extractions to combine")

## Summary

### Key Takeaways

1. **PDFs Are Challenging** - Extraction quality varies by report structure; need robust parsing

2. **Hybrid Approach Works Best** - Combine table parsing with LLM narrative extraction

3. **Validation Is Critical** - Always verify extracted numbers against known ranges

4. **Pipeline Automation Enables Scale** - Once built, can process years of historical data

5. **Store for Reuse** - Structured storage enables backtesting and analysis

### Skills Gained

✅ Downloading reports from predictable URL patterns  
✅ Extracting text and tables from PDFs  
✅ Parsing semi-structured table data  
✅ Using LLMs for narrative insight extraction  
✅ Building end-to-end data pipelines  
✅ Creating time series databases  

### What's Next

In **Notebook 2** (`02_usda_extraction.ipynb`), we'll apply similar techniques to USDA WASDE reports for agricultural commodities:
- Different document structure (tables embedded in Word/PDF)
- More complex data (global supply/demand balances)
- Multi-commodity extraction (corn, soybeans, wheat)

### Additional Resources

- **EIA Weekly Reports:** https://www.eia.gov/petroleum/supply/weekly/
- **pdfplumber Documentation:** https://github.com/jsvine/pdfplumber
- **Commodity Trading Fundamentals:** Focus on inventory levels, production, and demand

---

**Next:** [02_usda_extraction.ipynb](./02_usda_extraction.ipynb) - USDA WASDE Report Processing